# Predicting Credit Risk
## 1. Project Purpose
### Research Question
A bank that loans money to the wrong people can lose a lot of money. According to Shmueli et al. (Table 21.5), a good loan earns the bank 100 Deutsche Marks (DM) on average, while a bad loan costs 500 DM on average, which is five times as much. The purpose of this project is to analyze applicants' data to answer the following question: 

**Can we predict who will repay, so the bank approves the right people and loses less money?** 

### Goals
1. **Find the drivers:** identify which applicant traits are linked to bad credit
2. **Build a classifier:** train models that label new applicants as good or bad
3. **Maximize profit:** choose the model with the highest net profit

This report (Part 1) describes the dataset and begins to address goal 1 by exploring which applicants traits are linked to bad credit. It also prepares the data for modeling by checking for missing values, duplicates, and outliers, removing the ID column, and encoding categorical variables.

### Setup
Load the required libraries and the dataset.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data = pd.read_csv('../data/GermanCredit.csv')

## 2. Describe the Data
### German Credit Dataset
This project uses the German Credit dataset as provided in Shmueli et al. (Chapter 21), adapted from the Statlog German Credit data in the UCI Machine Learning Repository. The origianl version is from the 20th century, when humans had to label records as either good or poor credit.
### Structure & Size
This dataset has 32 variables - 30 predictors, 1 target (RESPONSE), and one ID column (OBS#) - and 1,000 records. Each row is a prior applicant for credit. Each applicant was rated as "good credit", coded as 1, or "bad credit", coded as 0.

In [4]:
# number of columns and rows
data.shape

(1000, 32)

In [5]:
# first 5 records
data.head()

,OBS#,CHK_ACCT,DURATION,HISTORY,NEW_CAR,USED_CAR,FURNITURE,RADIO/TV,EDUCATION,RETRAINING,...,AGE,OTHER_INSTALL,RENT,OWN_RES,NUM_CREDITS,JOB,NUM_DEPENDENTS,TELEPHONE,FOREIGN,RESPONSE
0,1,0,6,4,0,0,0,1,0,0,...,67,0,0,1,2,2,1,1,0,1
1,2,1,48,2,0,0,0,1,0,0,...,22,0,0,1,1,2,1,0,0,0
2,3,3,12,4,0,0,0,0,1,0,...,49,0,0,1,1,1,2,0,0,1
3,4,0,42,2,0,0,1,0,0,0,...,45,0,0,0,1,2,2,0,0,1
4,5,0,24,3,1,0,0,0,0,0,...,53,0,0,0,2,2,2,0,0,0


### Target Variable

In [10]:
# target variable balance
counts = data['RESPONSE'].value_counts()
percentage = data['RESPONSE'].value_counts(normalize=True) * 100

pd.DataFrame({'Count' : counts, 'Percentage (%)' : percentage})

,Count,Percentage (%)
RESPONSE,,
1,700,70.0
0,300,30.0


Out of the 1,000 records, 700 (70%) were labeled as good credit and 300 (30%) as bad credit. This imbalance means accuracy can be misleading: a model that approved every applicant would be 70% accurate, yet the bank would lose 80,000 DM (700 × 100 − 300 × 500). Because of this imbalance, the data will also be split with stratification, so that both the training and validation sets keep the 70%/30% proportion.

### Variables

In [6]:
# columns' name, non-null count, and data type
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 32 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   OBS#              1000 non-null   int64
 1   CHK_ACCT          1000 non-null   int64
 2   DURATION          1000 non-null   int64
 3   HISTORY           1000 non-null   int64
 4   NEW_CAR           1000 non-null   int64
 5   USED_CAR          1000 non-null   int64
 6   FURNITURE         1000 non-null   int64
 7   RADIO/TV          1000 non-null   int64
 8   EDUCATION         1000 non-null   int64
 9   RETRAINING        1000 non-null   int64
 10  AMOUNT            1000 non-null   int64
 11  SAV_ACCT          1000 non-null   int64
 12  EMPLOYMENT        1000 non-null   int64
 13  INSTALL_RATE      1000 non-null   int64
 14  MALE_DIV          1000 non-null   int64
 15  MALE_SINGLE       1000 non-null   int64
 16  MALE_MAR_or_WID   1000 non-null   int64
 17  CO-APPLICANT      1000 non-null   int64
 18  

In [15]:
# number of unique values per column
data.nunique()

OBS#                1000
CHK_ACCT               4
DURATION              33
HISTORY                5
NEW_CAR                2
USED_CAR               2
FURNITURE              2
RADIO/TV               2
EDUCATION              2
RETRAINING             2
AMOUNT               921
SAV_ACCT               5
EMPLOYMENT             5
INSTALL_RATE           4
MALE_DIV               2
MALE_SINGLE            2
MALE_MAR_or_WID        2
CO-APPLICANT           2
GUARANTOR              2
PRESENT_RESIDENT       4
REAL_ESTATE            2
PROP_UNKN_NONE         2
AGE                   53
OTHER_INSTALL          2
RENT                   2
OWN_RES                2
NUM_CREDITS            4
JOB                    4
NUM_DEPENDENTS         2
TELEPHONE              2
FOREIGN                2
RESPONSE               2
dtype: int64

All columns are stored as integers, but they represent four different types of variables (Table 21.3):

- **Binary (0/1):** NEW_CAR, USED_CAR, FURNITURE, RADIO/TV, EDUCATION, RETRAINING, MALE_DIV, MALE_SINGLE, MALE_MAR_or_WID, CO-APPLICANT, GUARANTOR, REAL_ESTATE, PROP_UNKN_NONE, OTHER_INSTALL, RENT, OWN_RES, TELEPHONE, FOREIGN. These are used as they are.
- **Categorical codes:** CHK_ACCT, HISTORY, SAV_ACCT, JOB. The codes are labels, not amounts, so these will be one-hot encoded.
- **Ordered codes:** EMPLOYMENT, PRESENT_RESIDENT, INSTALL_RATE. Higher codes mean more, so these are treated as numbers.
- **Numeric:** AMOUNT, DURATION, AGE, NUM_CREDITS, NUM_DEPENDENTS. These are used as they are.

Together, these make up the 30 predictors.

### Summary Statistics

In [12]:
# summary statistics
numeric_cols = ['AMOUNT', 'DURATION', 'AGE']
data[numeric_cols].describe().round(2)

,AMOUNT,DURATION,AGE
count,1000.00,1000.00,1000.00
mean,3271.26,20.90,35.55
std,2822.74,12.06,11.38
min,250.00,4.00,19.00
25%,1365.50,12.00,27.00
50%,2319.50,18.00,33.00
75%,3972.25,24.00,42.00
max,18424.00,72.00,75.00


The typical loan is 2,320 DM over 18 months, and the typical applicant is 33 years old (medians). All three variables are right-skewed: the mean is higher than the median in each case, most clearly for AMOUNT, where the mean (3,271 DM) is about 950 DM above the median. The maximums are also far above the 75th percentiles: a loan of 18,424 DM compared with 3,972 DM, a duration of 72 months compared with 24, and an age of 75 compared with 42. This suggests a few unusually large loans, long durations, and older applicants. Section 3 checks whether these are outliers using box plots.

I check the distribution of foreign workers, a sensitive variable discussed in the biases below.

In [14]:
# foreign worker distribution (1 = yes, 0 = no)
data['FOREIGN'].value_counts()

FOREIGN
0    963
1     37
Name: count, dtype: int64

### Assumptions, Limitations, and Biases

**Assumptions**

- **Labels are accurate:** RESPONSE was assigned based on a person's judgment, and the analysis assumes the ratings are correct. If they contain errors, the models will learn and repeat them.
- **Profit values are fixed:** the +100 DM and −500 DM figures are averages, but the analysis applies them to every loan, even though real gains and losses depend on the loan amount.
- **Codes are meaningful:** ordered variables such as EMPLOYMENT and INSTALL_RATE are treated as numbers, which assumes each step between codes represents a real increase.

**Limitations**

- **Old and local data:** the records come from a single German bank, collected awhile ago and recorded in Deutsche Marks, so the patterns may not apply to today's borrowers or to other countries.
- **Small sample:** with only 1,000 applicants, and 300 bad ones, complex models such as neural networks may overfit.
- **Imbalanced classes:** as shown above, 70% of applicants are good credit, so accuracy alone is a misleading measure of model performance.

**Biases**

- **Gender coding:** sex and marital status are combined into three variables for men only (MALE_DIV, MALE_SINGLE, MALE_MAR_or_WID). Women have no category of their own, so the data separates men by marital status but records no marital information for women.
- **Foreign workers:** according to Table 21.3, FOREIGN = 1 indicates a foreign worker, and only a small share of applicants fall into that group. The models will therefore learn little about foreign workers, and using this variable in lending decisions raises fairness concerns.
- **Age:** AGE may be a strong predictor, but rejecting applicants because they are young or old is a fairness concern.
- **Human judgment:** because the labels come from human raters, any bias in how they judged certain groups is built into the data.

In real lending, decisions based on sex, marital status, age, or national origin are restricted by fair-lending laws, so these variables require careful interpretation.
